In [ ]:
import sys
from pathlib import Path

_HERE = Path.cwd() if '__file__' not in globals() else Path(__file__).resolve().parent
_ROOT = _HERE.parents[1]
sys.path.insert(0, str(_HERE))

import numpy as np
import mujoco
import mujoco.viewer
from scipy.spatial import ConvexHull
from dataclasses import dataclass

from core import FloatingBaseRobotState, Pinocchio_Wrapper, Mujoco_Kernel
from utils import SimScheduler, Visualizer, DataLogger, plot_helpers
from control import WholeBodyController, MotionPlanner, TaskSpaceController, GaitScheduler, TrajOptimizer


%load_ext autoreload
%autoreload 2

URDF = str(_ROOT / 'models' / 'g1' / 'urdf' / 'g1_29dof.urdf')
PKG  = str(_ROOT / 'models' / 'g1')
MJCF = str(_ROOT / 'models' / 'scenes' / 'g1' / 'scene_29dof.xml')

In [ ]:
PHASE_OFFSET  = np.array([0.0, 0.5])   # LEG_KEYS 순서: L_foot=0, R_foot=1
HEIGHT_SWING  = 0.08                   # swing foot peak height [m]
LEG_KEYS      = ["L_foot", "R_foot"]
LEG_IDX       = {k: i for i, k in enumerate(LEG_KEYS)}

In [ ]:
STANDING_MODE = True   # True: 양발 항상 stance, gait 비활성 / False: 보행 모드

class RobotOrchestrator:
    def __init__(self, kernel, wrapper, state, controller, planner, gait, wbc, logger=None):
        self.kernel     = kernel
        self.wrapper    = wrapper
        self.state      = state
        self.controller = controller
        self.planner    = planner
        self.gait       = gait
        self.wbc        = wbc
        self.logger     = logger

        self.swing_pos_des   = {leg: np.zeros(3) for leg in LEG_KEYS}
        self.swing_pos_start = {
            leg: wrapper.get_ee_state_world(leg)[0].translation.copy()
            for leg in LEG_KEYS
        }
        self._prev_contact = np.ones(len(LEG_KEYS), dtype=int)

    def on_control(self, t):
        # 상태 갱신
        self.kernel.update_robot_state(self.state)
        self.wrapper.update_model(self.state.q, self.state.dq)

        if STANDING_MODE:
            contact_mask  = np.ones(len(LEG_KEYS), dtype=int)
            tau_swing     = np.zeros(self.wrapper.na)
        else:
            contact_mask, swing_phases = self.gait.compute_gait_state(t)

            for i, leg in enumerate(LEG_KEYS):
                if self._prev_contact[i] == 1 and contact_mask[i] == 0:
                    oMf, _, _ = self.wrapper.get_ee_state_world(leg)
                    self.swing_pos_start[leg] = oMf.translation.copy()
            self._prev_contact[:] = contact_mask

            J_c_stance = self._get_stance_jacobian(contact_mask)

            tau_swing = np.zeros(self.wrapper.na)
            for i, leg in enumerate(LEG_KEYS):
                if contact_mask[i] == 0:
                    pos_target = self.planner.compute_raibert_heuristic(leg=leg)
                    pos_des, vel_des, _ = self.planner.compute_bezier_trajectory(
                        swing_phases[i], self.swing_pos_start[leg],
                        pos_target, self.gait.swing_time, HEIGHT_SWING)
                    self.swing_pos_des[leg] = pos_des
                    tau_swing += self._swing_task_torque(leg, pos_des, vel_des, J_c_stance)

        # WBC: balance torque — 고정 기준점으로 복원력 확보
        tau_wbc = self.wbc.compute(COM_POS_DES, np.zeros(3), contact_mask=contact_mask)
        tau     = tau_wbc + tau_swing
        self.kernel.ctrl_tau = tau

        # 로깅 (standing/walking 공통)
        if self.logger is not None:
            F_hat = self.wbc.force_opt.opt_F  # (12,) [f_L,τ_L,f_R,τ_R]
            self.logger.log(
                t            = t,
                com_des      = COM_POS_DES,
                com_curr     = self.wrapper.com_pos_world,
                com_vel      = self.wrapper.com_vel_world,
                contact_mask = contact_mask,
                F_L          = F_hat[0:3],
                F_R          = F_hat[6:9],
                tau          = tau,
                q_act        = self.state.q[7:],   # actuated joints (free-flyer 7개 제외)
                dq_act       = self.state.dq[6:],
            )

    def _get_stance_jacobian(self, contact_mask):
        Js = [self.wrapper.get_J_world(leg)
              for i, leg in enumerate(LEG_KEYS) if contact_mask[i] == 1]
        return np.vstack(Js) if Js else None

    def _swing_task_torque(self, leg, pos_des, vel_des, J_c_stance):
        """swing 발 위치 추종 → null-space projected tau (na,)."""
        oMf, lin_vel, _ = self.wrapper.get_ee_state_world(leg)
        pos_err = pos_des - oMf.translation
        vel_err = vel_des - lin_vel

        J_p = self.wrapper.get_J_world(leg)[:3]
        Kp, Kd = self.controller.Kp_pos, self.controller.Kd_lin
        F_des  = Kp @ pos_err + Kd @ vel_err

        M_inv   = self.wrapper.M_inv
        JMinvJT = J_p @ M_inv @ J_p.T + self.controller.damping2 * np.eye(3)
        tau_pure = J_p.T @ np.linalg.solve(JMinvJT, F_des)

        if J_c_stance is not None:
            nc          = J_c_stance.shape[0]
            M_inv_tau   = M_inv @ tau_pure
            JcMinvJcT   = J_c_stance @ M_inv @ J_c_stance.T
            gamma       = np.linalg.solve(JcMinvJcT + 1e-6 * np.eye(nc),
                                          J_c_stance @ M_inv_tau)
            tau_pure    = tau_pure - J_c_stance.T @ gamma

        return tau_pure[6:]

    def on_render(self, t, visualizer):
        _, points = self.kernel.get_foot_contact_state()
        if len(points) >= 3:
            pts = np.array(points)
            try:
                hull  = ConvexHull(pts[:, :2])
                z     = pts[:, 2].mean()
                verts = pts[hull.vertices]
                n     = len(verts)
                for j in range(n):
                    p1 = np.array([verts[j][0],       verts[j][1],       z])
                    p2 = np.array([verts[(j+1)%n][0], verts[(j+1)%n][1], z])
                    visualizer.draw_line(p1, p2, rgba=(0, 1, 0, 1), thickness=0.005)
            except Exception:
                pass
        for leg in LEG_KEYS:
            visualizer.draw_trajectory(self.swing_pos_des[leg], rgba=(1.0, 0.5, 0.0, 1.0))

상태 초기화, wrapper 호출, 커널 호출, 제어기 호출, 초기 ee SE3, SO3, x, y, z 추출 (피노키오 컨벤션)

In [ ]:
state    = FloatingBaseRobotState()
wrapper  = Pinocchio_Wrapper(URDF, PKG)
joint_names = [wrapper.model.names[i] for i in range(2, wrapper.model.njoints)]
kernel   = Mujoco_Kernel(MJCF, joint_names_pin_order=joint_names)
kernel.register_foot_geoms({"L_foot": "left_ankle_roll_link",
                             "R_foot": "right_ankle_roll_link"})

controller = TaskSpaceController(wrapper)
gait       = GaitScheduler(gait_hz=1.0, duty=0.7)
optimizer  = TrajOptimizer(wrapper)
planner    = MotionPlanner(wrapper, optimizer, gait)
wbc        = WholeBodyController(wrapper)

print("✅ 시스템 호출 완료")

# 키프레임에서 Pin 포맷으로 정확히 변환해서 기준값 추출
kernel.reset_to_keyframe()
kernel.update_robot_state(state)
wrapper.update_model(state.q, np.zeros(wrapper.nv))

COM_HEIGHT_DES = float(wrapper.com_pos_world[2])
COM_POS_DES    = wrapper.com_pos_world.copy()   # 수평 기준도 고정
print(f"COM_POS_DES = {COM_POS_DES}")

init_oMb      = wrapper.oMb
init_oM_Lfoot = wrapper.oM_Lfoot
init_oM_Rfoot = wrapper.oM_Rfoot
init_oM_Lhand = wrapper.oM_Lhand
init_oM_Rhand = wrapper.oM_Rhand

Main Loop (상태 갱신, 제어, 뷰어, 시각화)

In [ ]:
# 실행 직전 초기화
kernel.reset_to_keyframe()
logger = DataLogger()

# 오케스트레이터 및 스케줄러 실행
orch  = RobotOrchestrator(kernel, wrapper, state, controller, planner, gait, wbc, logger=logger)
sched = SimScheduler(kernel.model, kernel.data, ctrl_hz=500, render_hz=60)
sched.run(on_control=orch.on_control, on_render=orch.on_render)

In [ ]:
import matplotlib.pyplot as plt

# 로그 추출
times    = logger.get('t')
com_des  = logger.get('com_des')         # (N, 3)
com_curr = logger.get('com_curr')        # (N, 3)
com_vel  = logger.get('com_vel')         # (N, 3)
F_L      = logger.get('F_L')             # (N, 3)
F_R      = logger.get('F_R')             # (N, 3)
tau      = logger.get('tau')             # (N, na)
mask     = logger.get('contact_mask')    # (N, n_legs)
q_act    = logger.get('q_act')           # (N, na)
dq_act   = logger.get('dq_act')          # (N, na)

mass_g = wrapper.mass * 9.81
print(f"총 step={len(times)}, 모드={'STANDING' if STANDING_MODE else 'WALKING'}, mg={mass_g:.1f}N")

# 1. CoM 추종 (XYZ 오차)
plot_helpers.plot_ee_tracking(times, com_des, com_curr,
                              title="CoM Position Tracking (des - curr)")

# 2. CoM 속도 (정지 상태에서 0 근방 유지 확인)
plot_helpers.plot_velocity_tracking(times, np.zeros_like(com_vel), com_vel,
                                    title="CoM Velocity (des=0)")

# 3. Contact force (fx, fy, fz; fz 합 ≈ mg 이어야 함)
#    plot_contact_schedule 시그니처: contact_mask (n_legs, N), force (3*n_legs, N)
contact_mat = mask.T                                                # (n_legs, N)
force_mat   = np.vstack([F_L.T, F_R.T])                              # (6, N)
dt = float(np.mean(np.diff(times))) if len(times) > 1 else 1/500
plot_helpers.plot_contact_schedule(times, contact_mat, force_mat, dt,
                                   leg_names=LEG_KEYS,
                                   title="Contact Forces (QP solution)")

# 4. fz 합 vs mg
fig, ax = plt.subplots(figsize=(12, 3.5))
ax.plot(times, F_L[:, 2], label='fz_L')
ax.plot(times, F_R[:, 2], label='fz_R')
ax.plot(times, F_L[:, 2] + F_R[:, 2], 'k-', linewidth=2, label='fz_L + fz_R')
ax.axhline(mass_g, color='r', linestyle='--', label=f'mg = {mass_g:.1f} N')
ax.set_xlabel('Time [s]'); ax.set_ylabel('Vertical Force [N]')
ax.set_title('Vertical Contact Force vs Gravity'); ax.grid(True); ax.legend()
plt.tight_layout(); plt.show()

# 5. Joint torque
plot_helpers.plot_joint_state(times, tau, dq_act,
                              joint_names=joint_names,
                              title="Torque (top) & Joint Velocity (bottom)")